# Exercise 1: Basic ReAct Agent with Bielik

In this exercise you'll create your first agent using NeMo Agent Toolkit (NAT).

The agent uses the **ReAct** (Reasoning + Acting) pattern:
1. **Observe** the user's question
2. **Think** about which tool to use
3. **Act** by calling a tool
4. **Observe** the result and repeat until done

```
User Question --> ReAct Agent (Bielik) --> Tool Call --> Tool Result --> Final Answer
                      |                      |
                      +-- wiki_search --------+
                      +-- current_datetime ---+
```

This is a ReAct (Reasoning and Acting) agent, based on the [ReAct paper](https://react-lm.github.io/). The ReAct agent’s prompt is directly inspired by the prompt examples in the appendix of the paper. The agent uses the NVIDIA NeMo Agent Toolkit core library agents and tools to perform ReAct reasoning between tool calls. In your YAML config files, you can customize prompts for your specific needs.

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"Endpoint: {os.getenv('VLLM_BASE_URL')}")
print(f"Model:    {os.getenv('VLLM_MODEL_NAME')}")

Endpoint: https://bielik-3-7b-minitron-artur-1-runai-de-beyond-pr1.fin2work.runai.beyond.pl/v1
Model:    speakleash/Bielik-Minitron-7B-v3.0-Instruct


## Step 1: Explore the YAML configuration

Open `config.yaml` and examine the three sections:
- **llms** - Defines the Bielik model endpoint
- **functions** - `wiki_search` (Wikipedia) and `current_datetime` tools
- **workflow** - ReAct agent connecting the LLM to tools

In [11]:
with open("config.yaml") as f:
    print(f.read())

# =============================================================
# Exercise 1: Basic ReAct Agent with Bielik
# =============================================================
# This config defines a simple ReAct agent that uses Bielik
# served via vLLM to answer questions using tools.
# =============================================================

llms:
  bielik:
    _type: openai
    base_url: "${VLLM_BASE_URL}"
    model: "${VLLM_MODEL_NAME}"
    api_key: "${VLLM_API_KEY}"
    temperature: 0.7
    max_tokens: 2048

functions:
  wiki_search:
    _type: wiki_search
    max_results: 2

  current_time:
    _type: current_datetime

workflow:
  _type: react_agent
  tool_names:
    - wiki_search
    - current_time
  llm_name: bielik
  max_tool_calls: 5
  verbose: true
  parse_agent_response_max_retries: 3
  raise_on_parsing_failure: false
  additional_instructions: "Używaj jak najmniej wywołań narzędzi. Gdy masz wystarczająco informacji, natychmiast podaj Final Answer. Nie szukaj dodatkowych 

# Understanding the workflow

In NeMo Agent Toolkit, a workflow defines which functions and models are used to perform a given task or series of tasks. A workflow definition is specified in a YAML configuration file. The workflow section of the configuration file defines the workflow itself, and specifies a function, typically an agent, which will orchestrate which functions and models are called to complete the given task.

The **functions** section contains the tools used in the **workflow**, while **llms** define the models used in the workflow, and lastly the **workflow** section ties the other sections together and defines the workflow itself.

The workflow itself is typically an agent, however any NeMo Agent Toolkit function can be used as a workflow. 

## LLMs

The LLM configuration is defined in the **llms** section of the workflow configuration file. The **_type** value refers to the LLM provider, and the model_name value always refers to the name of the model to use.

```
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
  openai_llm:
    _type: openai
    model_name: gpt-4o-mini
  aws_bedrock_llm:
    _type: aws_bedrock
    model_name: meta/llama-3.1-70b-instruct
    region_name: us-east-1
  azure_openai_llm:
    _type: azure_openai
    azure_deployment: gpt-4o-mini
  litellm_llm:
    _type: litellm
    model_name: gpt-4o
  huggingface_llm:
    _type: huggingface
    model_name: Qwen/Qwen3Guard-Gen-0.6B
```

### OpenAI

You can use the following environment variables to configure the OpenAI LLM provider:

`OPENAI_API_KEY` - The API key to access OpenAI resources

The OpenAI LLM provider is defined by the OpenAIModelConfig class.

**model_name** - The name of the model to use  
**temperature** - The temperature to use for the model  
**top_p** - The top-p value to use for the model  
**max_tokens** - The maximum number of tokens to generate  
**seed** - The seed to use for the model  
**api_key** - The API key to use for the model  
**base_url** - The base URL to use for the model  
**max_retries** - The maximum number of retries for the request  
**request_timeout** - HTTP request timeout in seconds  

## Functions

Functions are reusable components that perform specific operations, such as web searches, API calls, or calculations.

In NeMo Agent Toolkit, functions are a core abstraction that offer type-safe, asynchronous operations with support for both single and streaming outputs. They wrap callable objects (like Python functions or coroutines) and enhance them with:

- Type validation and conversion
- Schema-based input/output validation via Pydantic models
- Unified interfaces to improve composability
- Support for both streaming and non-streaming (single) outputs

### Included Functions

For a complete list of functions run the following command:
`nat info components -t function`

In [12]:
!nat info components -t function

                               NAT Search Results                               
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ package        ┃ version  ┃ component_type ┃ component_name ┃ description    ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ nvidia-nat-lan │ 1.6.0rc1 │ function       │ langgraph_wrap │ Configuration  │
│ gchain         │          │                │ per            │ model for the  │
│                │          │                │                │ LangGraph      │
│                │          │                │                │ wrapper.       │
│                │          │                │                │                │
│                │          │                │                │   Args:        │
│                │          │                │                │     _type      │
│                │          │                │                │ (str): The     │
│                │          

## Function Groups

Function groups let you package multiple related functions together so they can share configuration, context, and resources within the NVIDIA NeMo Agent Toolkit.



# Other Prebuild Agents apart from React

For a complete list please refer to docuentation

- Reasoning Agent
```yaml
workflow:
  _type: reasoning_agent
  llm_name: nemotron_model
  # The augmented_fn is the nat Function that the execution plan is passed to. Usually an agent entry point.
  augmented_fn: react_agent
  verbose: true
```

- ReWOO Agent
```yaml
workflow:
  _type: rewoo_agent
  tool_names: [wikipedia_search, current_datetime, code_generation, math_agent]
  llm_name: nim_llm
  verbose: true
  use_tool_schema: true
```

- Router Agent
```yaml
workflow:
  _type: router_agent
  branches: [fruit_advisor, city_advisor, literature_advisor]
  llm_name: nim_llm
  detailed_logs: true
```

- Tool Calling Agent
```yaml
workflow:
  _type: tool_calling_agent
  tool_names: [wikipedia_search, current_datetime, code_generation]
  llm_name: nim_llm
  verbose: true
  handle_tool_errors: true
```



The **_type** value refers to the workflow type, in our example we are using a react_agent workflow. 

## Override

Each workflow contains several configuration parameters that can be modified to customize the workflow. While copying and modifying the file is possible, it is not always necessary as some parameters can be overridden using the `--override` flag.

Usage:
`nat run --config_file examples/getting_started/simple_web_query/configs/config.yml --input "What is LangSmith?"  \
  --override llms.nim_llm.temperature 0.7`

The --override flag can be specified multiple times, allowing the ability to override multiple parameters. 

## Step 2: Run the agent

Watch the ReAct loop: Thought -> Action -> Observation -> repeat

In [13]:
!nat run --config_file config.yaml --input "Kim był Mikołaj Kopernik i jaki był jego główny wkład w naukę?"

2026-04-13 18:25:04 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-04-13 18:25:05 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-04-13 18:25:06 - INFO     - httpx:1740 - HTTP Request: POST https://bielik-3-7b-minitron-artur-1-runai-de-beyond-pr1.fin2work.runai.beyond.pl/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 18:25:06 - INFO     - nat.plugins.langchain.agent.react_agent.agent:236 - 
------------------------------
[AGENT]
Agent input: Kim był Mikołaj Kopernik i jaki był jego główny wkład w naukę?
Agent's thoughts: 
Action: wiki_search
Action Input: "Kim był Mikołaj Kopernik i jaki był jego główny

In [14]:
!nat run --config_file config.yaml --input "Jaka jest aktualna data i godzina? Czym jest Wisła?"

2026-04-13 18:25:12 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-04-13 18:25:13 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-04-13 18:25:13 - INFO     - httpx:1740 - HTTP Request: POST https://bielik-3-7b-minitron-artur-1-runai-de-beyond-pr1.fin2work.runai.beyond.pl/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 18:25:13 - INFO     - nat.plugins.langchain.agent.react_agent.agent:236 - 
------------------------------
[AGENT]
Agent input: Jaka jest aktualna data i godzina? Czym jest Wisła?
Agent's thoughts: 
I know the final answer
Final Answer: Aktualna data i godzina to [wynik z current_time], Wisł

## Step 3: Try your own queries

Experiment with different questions. Notice how the agent decides which tool to use.

In [15]:
!nat run --config_file config.yaml --input "TWOJE PYTANIE TUTAJ"

2026-04-13 18:25:16 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-04-13 18:25:16 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-04-13 18:25:17 - INFO     - httpx:1740 - HTTP Request: POST https://bielik-3-7b-minitron-artur-1-runai-de-beyond-pr1.fin2work.runai.beyond.pl/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 18:25:17 - INFO     - nat.plugins.langchain.agent.react_agent.agent:236 - 
------------------------------
[AGENT]
Agent input: TWOJE PYTANIE TUTAJ
Agent's thoughts: 
I'm sorry, but it seems like there's no specific question provided for me to answer. Could you please provide the question yo

# Create a New Tool and Workflow with NVIDIA NeMo Agent Toolkit

In this **workflow** we have been primarily utilizing tools that were included with the NeMo Agent Toolkit. To create a new workflow or tool we can run a cli nat tool to create empty component.

`nat workflow create --workflow-dir examples bielik_agent`

This command does the following:

- Creates a new directory, `examples/bielik_agent`.
- Sets up the necessary files and folders.
- Installs the new Python package for your workflow.

Each workflow created in this way also creates a Python project, and by default, this will also install the project into the environment. 

This creates a new directory `examples/bielik_agent` with the following layout:

```
examples/bielik_agent
├── configs -> src/bielik_agent/configs
├── data -> src/bielik_agent/data
├── pyproject.toml
└── src
    └── bielik_agent
        ├── __init__.py
        ├── configs
        │   └── config.yml
        ├── data
        ├── register.py
        └── bielik_agent.py
```

## Key Takeaways

- NAT workflows are defined declaratively in YAML
- The `_type: openai` LLM works with any OpenAI-compatible endpoint (vLLM, NIM, etc.)
- ReAct agents reason step-by-step before calling tools
- `nat run` is the simplest way to execute a workflow

**Next:** We'll build more complex agents with LangChain and CrewAI, then register them as NAT Functions.